# LogSpace capacity analytics & forecast

**Input:** Excel master file `METRICS_XLSX`: rows of the daily `logspace_metrics.csv` produced by
`collect_logspace_metrics.sh`, appended by hand
(events per category, partition sizes from `backup_tool --cmd listpart`, `df` snapshots, rotation health).

**Export:** every chart is saved to `EXPORT_DIR` as an interactive `<chart>_<last complete day>.html`, every table
as `<table>_<last complete day>.xlsx` — files of different days can be opened side by side and compared.
Each chart is followed by a short *How to read* note.

**Daily routine:** `Kernel → Restart & Run All`, then read **section 10 (Daily summary)** first,
and go to the section a WARN/CRIT points to. Full methodology: `METHODOLOGY.md`.

| Section | Question it answers |
|---|---|
| 1 | Is the input data complete and fresh? |
| 2 | How many bytes does one event really cost (D)? |
| 3 | Do `listpart` bytes match the physical disk? |
| 4 | Which categories grow and how fast? |
| 5 | Which categories are expensive on disk? |
| 6 | Model backtest: which forecast to trust |
| 7 | Daily volume forecast (bytes and EPS) |
| 8 | When rotation starts, retention depth, physical disk free |
| 9 | What-if: flow changes |
| 10 | Daily summary with statuses |

## 0. Settings

In [ ]:
from pathlib import Path

METRICS_XLSX  = Path(r"I:\SHARE\AN\metrics.xlsx")  # master file: daily CSV rows appended by hand
METRICS_SHEET = 0           # sheet name or index (0 = first sheet)
LEGACY_XLSX   = None        # e.g. Path("eps.xlsx"): old Excel history (sheet RAW), merged under collector data
LEGACY_SHEET  = "RAW"

DBSPACE       = 0.85        # TailcutterDbspace as a fraction of total disk
TTL_DAYS      = 365         # TailcutterTtl
DISK_TOTAL_OVERRIDE = None  # bytes; None = take total from the latest df snapshot

ANOMALY_RATIO = 0.5         # day below 50% of rolling median = collection outage
TREND_WINDOW  = 90          # days of history for linear trends
BACKTEST_DAYS = 28          # holdout used to compare forecast models
HORIZON       = 180         # forecast horizon, days
VENDOR_D      = 200         # vendor formula coefficient for LogSpace, bytes/event
EXPORT_DIR    = Path("export")  # charts (.html) and tables (.xlsx) as <name>_<last complete day> (real data - never commit)

SARIMAX_WINDOW         = 180            # days of history SARIMAX is fitted on (None = all)
SARIMAX_ORDER          = (1, 1, 1)      # (p, d, q)
SARIMAX_SEASONAL_ORDER = (1, 0, 1, 7)   # (P, D, Q, period): weekly cycle

# Thresholds for the daily summary
WARN_DAYS_TO_LIMIT = 30     # rotation will start in fewer days -> WARN
WARN_VOLUME_DEV    = 0.30   # yesterday vs same-weekday baseline
WARN_CATEGORY_DEV  = 0.50   # per-category deviation vs same-weekday baseline
WARN_D_DRIFT       = 0.10   # D vs 28-day median
WARN_TC_LOG_AGE    = 90     # minutes since tailcutter log was written

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize, nnls
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

GiB, TiB = 1024**3, 1024**4
DAY_MS = 86_400_000      # one day on a plotly date axis
REPORT_DAY = None        # last complete day, set in section 1; used in export file names
HOVER_DAY = "%d.%m.%Y, %a"


def gb(x):
    """Bytes -> GiB."""
    return x / GiB


def style(fig, title, ytitle, height=470, daily_ticks=False, slider=False):
    """Common look for charts over dates.
    The hover box shows the exact day (with weekday) and every line's value on it; a dotted vertical cursor
    follows the mouse. daily_ticks: a label under every day (short ranges). Otherwise labels get finer when
    you zoom in; slider adds a range slider and 1-month / 3-month / all buttons."""
    fig.update_layout(template="plotly_white", title=title, height=height, yaxis_title=ytitle,
                      hovermode="x unified", margin=dict(l=60, r=20, t=70, b=40),
                      legend=dict(orientation="h", yanchor="top", y=-0.3 if slider else -0.2))
    fig.update_xaxes(hoverformat=HOVER_DAY, showspikes=True, spikemode="across", spikesnap="cursor",
                     spikethickness=1, spikedash="dot", spikecolor="grey")
    if daily_ticks:
        xs = pd.DatetimeIndex([x for t in fig.data if t.x is not None and len(t.x) for x in (t.x[0], t.x[-1])])
        fig.update_xaxes(dtick=DAY_MS, tickformat="%d.%m", tickangle=-90,
                         range=[xs.min() - pd.Timedelta(hours=12), xs.max() + pd.Timedelta(hours=12)])
    else:
        fig.update_xaxes(tickformatstops=[dict(dtickrange=[None, 40 * DAY_MS], value="%d.%m"),
                                          dict(dtickrange=[40 * DAY_MS, None], value="%m.%Y")])
    if slider:
        fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.06),
                         rangeselector=dict(buttons=[dict(count=1, label="1 mo", step="month", stepmode="backward"),
                                                     dict(count=3, label="3 mo", step="month", stepmode="backward"),
                                                     dict(step="all", label="all")]))
    return fig


def weekends(fig, index, row=None, col=None):
    """Shade Saturday-Sunday, so weekly dips can be matched to weekends at a glance.
    Skipped on long ranges (over 120 days), where the bands would turn into stripes."""
    index = pd.DatetimeIndex(index)
    if len(index) == 0 or (index.max() - index.min()).days > 120:
        return
    for d in index[index.dayofweek == 5]:
        fig.add_vrect(x0=d - pd.Timedelta(hours=12), x1=d + pd.Timedelta(hours=36), fillcolor="lightgrey",
                      opacity=0.3, line_width=0, layer="below", row=row, col=col)


def mark(fig, when, label="last complete day"):
    """Vertical line separating history from forecast (single-panel charts)."""
    fig.add_shape(type="line", x0=when, x1=when, y0=0, y1=1, yref="paper",
                  line=dict(color="black", dash="dot", width=1))
    fig.add_annotation(x=when, y=0.99, yref="paper", text=label, showarrow=False, xanchor="left", yanchor="top",
                       font=dict(size=11), bgcolor="rgba(255,255,255,0.7)")


def line(x, y, name, unit, **kw):
    """Line trace whose hover shows the value with its unit."""
    return go.Scatter(x=x, y=y, mode=kw.pop("mode", "lines"), name=name,
                      hovertemplate="%{y:,.1f} " + unit, **kw)


def out(fig, name):
    """Save the chart as EXPORT_DIR/<name>_<REPORT_DAY>.html and show it.
    plotly.js is written once next to the files, so they open offline in any browser."""
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    path = EXPORT_DIR / f"{name}_{REPORT_DAY:%Y-%m-%d}.html"
    fig.write_html(path, include_plotlyjs="directory")
    print(f"Saved: {path}")
    fig.show()


def out_table(df, name, show=True):
    """Save the table as EXPORT_DIR/<name>_<REPORT_DAY>.xlsx and display it."""
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    path = EXPORT_DIR / f"{name}_{REPORT_DAY:%Y-%m-%d}.xlsx"
    df.to_excel(path)
    print(f"Saved: {path}")
    if show:
        display(df)

## 1. Load data and quality check

Duplicates (same `record_type`, `metric_date`, `key`) are resolved by the latest `collected_at`,
so re-running the collector is safe. Partitions deleted by rotation keep their last known size,
so the volume history survives rotation.

In [ ]:
raw_all = (pd.read_excel(METRICS_XLSX, sheet_name=METRICS_SHEET, dtype={"key": str})
             .dropna(subset=["record_type"]))
# Excel may turn date strings into real dates on paste; accept both
raw_all["collected_at"] = pd.to_datetime(raw_all["collected_at"], dayfirst=True, format="mixed")
raw_all["metric_date"] = pd.to_datetime(raw_all["metric_date"], dayfirst=True, format="mixed")

# Optional legacy Excel history: lowest priority, collector rows override it
if LEGACY_XLSX is not None:
    lx = pd.read_excel(LEGACY_XLSX, sheet_name=LEGACY_SHEET)
    lx = pd.DataFrame({
        "collected_at": pd.Timestamp("1970-01-01"),
        "metric_date": pd.to_datetime(lx["Date"]),
        "record_type": "events",
        "key": lx["category"].astype(str),
        "value": lx["total_events_per_cat_daily"],
    })
    raw_all = pd.concat([lx, raw_all], ignore_index=True)

raw = (raw_all.sort_values("collected_at")
              .drop_duplicates(["record_type", "metric_date", "key"], keep="last"))

# --- latest snapshot: what is stored right now ---
part_all = raw_all[raw_all["record_type"] == "partition"]
last_snap = part_all["collected_at"].max()
snap_day = last_snap.normalize()
stored = part_all[part_all["collected_at"] == last_snap]
stored_events = stored[stored["key"] == "events"].set_index("metric_date")["value"].sort_index()
stored_other = stored.loc[stored["key"] != "events", "value"].sum()
stored_total = stored["value"].sum()

# --- disk snapshots (time series by collection time) ---
disk = (raw_all[raw_all["record_type"] == "disk"]
        .pivot_table(index="collected_at", columns="key", values="value", aggfunc="last")
        .sort_index())
part_tot_by_snap = part_all.groupby("collected_at")["value"].sum()
disk["partitions"] = part_tot_by_snap.reindex(disk.index)
DISK_TOTAL = DISK_TOTAL_OVERRIDE or (disk["total"].iloc[-1] if len(disk) else None)
LIMIT = DISK_TOTAL * DBSPACE if DISK_TOTAL else None

health = (raw[raw["record_type"] == "health"].sort_values("collected_at")
          .groupby("key")["value"].last())

# --- events per category (date x category) ---
ev_cat = (raw[raw["record_type"] == "events"]
          .pivot_table(index="metric_date", columns="key", values="value", aggfunc="sum")
          .sort_index().fillna(0))

# --- daily frame: complete days only (the snapshot day itself is still being written) ---
part_hist = raw[(raw["record_type"] == "partition") & (raw["key"] == "events")] \
               .set_index("metric_date")["value"]
last_day = min(part_hist.index.max(), snap_day - pd.Timedelta(days=1))
first_day = part_hist.index.min()
daily = pd.DataFrame(index=pd.date_range(first_day, last_day, freq="D"))
daily.index.name = "date"
daily["bytes"] = part_hist.reindex(daily.index)
daily["events"] = ev_cat.sum(axis=1).reindex(daily.index)
daily["eps"] = daily["events"] / 86400
daily["D"] = daily["bytes"] / daily["events"]

roll_med = daily["bytes"].rolling(15, center=True, min_periods=5).median()
daily["anomaly"] = daily["bytes"].isna() | (daily["bytes"] < roll_med * ANOMALY_RATIO)

# Clean continuous series for models: outage days are interpolated, not dropped
y_bytes = daily["bytes"].mask(daily["anomaly"]).interpolate(limit_direction="both")
REPORT_DAY = last_day

# --- quality report ---
missing_bytes = daily.index[daily["bytes"].isna()]
missing_events = daily.index[daily["events"].isna()]
print(f"Latest snapshot:      {last_snap:%d.%m.%Y %H:%M}")
print(f"Daily history:        {first_day:%d.%m.%Y} … {last_day:%d.%m.%Y} ({len(daily)} days)")
print(f"Stored now:           {len(stored_events)} event partitions, {stored_total/TiB:.2f} TiB")
print(f"Events data:          {daily['events'].notna().sum()} days, {ev_cat.shape[1]} categories")
print(f"Disk snapshots:       {len(disk)}")
print(f"Days without bytes:   {len(missing_bytes)}")
print(f"Days without events:  {len(missing_events)}  (backfill with: collect_logspace_metrics.sh -f … -t …)")
print("Outage days (excluded):", ", ".join(f"{d:%d.%m.%Y}" for d in daily.index[daily["anomaly"] & daily["bytes"].notna()]) or "none")

# --- daily volume: one bar per day (size of that day's events partition, latest known value) ---
real = ~daily["anomaly"]
fig = go.Figure()
fig.add_trace(go.Bar(x=daily.index[real], y=gb(daily.loc[real, "bytes"]), name="GiB per day",
                     marker_color="steelblue", hovertemplate="%{y:,.1f} GiB"))
fig.add_trace(go.Bar(x=daily.index[~real], y=gb(daily.loc[~real, "bytes"].fillna(0)), name="outage day (excluded)",
                     marker_color="crimson", hovertemplate="%{y:,.1f} GiB"))
fig.add_trace(line(daily.index, gb(y_bytes.rolling(7).mean()), "7-day mean", "GiB", line=dict(color="black")))
weekends(fig, daily.index)
style(fig, "Daily volume: size of each day's events partition", "GiB/day", slider=True)
out(fig, "daily_volume")

daily_tbl = pd.DataFrame({"GiB": gb(daily["bytes"]).round(2), "events": daily["events"],
                          "EPS": daily["eps"].round(0), "D, B/event": daily["D"].round(1),
                          "outage": daily["anomaly"]})
out_table(daily_tbl, "daily", show=False)

### How to read section 1

**The printed report**
- `Latest snapshot` — time of the newest collector run. Older than a day → the collector did not run (section 10).
- `Daily history` — days with partition sizes. `Stored now` — partitions on disk at the latest snapshot.
- `Events data` — days with per-category event counts. Sections 2, 4, 5 and the EPS charts need these; if they are
  missing, backfill them with `collect_logspace_metrics.sh -f … -t …`.
- `Outage days` — days below `ANOMALY_RATIO` of the rolling median. They are excluded from models and drawn red.

**The chart** — one bar per day = size of that day's events partition, GiB.
- Blue bars are real days, red bars are outage days (excluded), the black line is the 7-day mean.
- Grey bands are weekends: the regular dips should fall inside them.
- Hover shows the exact date with the weekday; use *1 mo / 3 mo / all* or the slider under the chart to zoom.
- What to look for: the level of the black line (today's typical day), steps in it (a source was added or lost),
  single tall bars (one-off events) and red bars (collection problems).

**The table** `daily_<day>.xlsx` (saved, not shown): per day — GiB, events, EPS, D and the outage flag.

*Example (illustrative):* the black line runs flat at ~40 GiB/day, then within a week climbs to ~60 and stays there
→ a step, not a trend: find which categories grew in section 4. A single 90 GiB bar on a Tuesday with normal days
around it is a one-off event; a red bar at 5 GiB on a Saturday is a collection outage, not a quiet weekend.

## 2. Real cost of one event (D)

**Left:** daily bytes vs daily events with a linear regression.
The slope is the marginal cost of one extra event; a tight cloud along the line means
"disk usage is driven by event count" and EPS-based forecasts are reliable.
**Right:** D over time. A step in D with flat EPS means the event mix changed
(heavier categories or longer raw messages), not the flow.

**When D is computed.** At least **7** usable days are needed. A day is usable when it is complete
(before the latest snapshot day), has `events` partition bytes from `listpart`, has events data and is
not an outage day. `D_FACT` is the median D over the last 28 usable days; with fewer than 28 it is
less stable (weekdays and weekends differ). With fewer than 7 days `D_FACT` falls back to `VENDOR_D`.

**Vendor value.** `VENDOR_D = 200` bytes/event: coefficient of the LogSpace sizing formula in the
PT MaxPatrol SIEM documentation (exact document version and section not recorded here).

In [ ]:
dd = daily.loc[~daily["anomaly"] & daily["events"].notna()]
HAS_D = len(dd) >= 7

if HAS_D:
    slope, icpt = np.polyfit(dd["events"], dd["bytes"], 1)
    r2 = np.corrcoef(dd["events"], dd["bytes"])[0, 1] ** 2
    D_FACT = dd["D"].tail(28).median()

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                        subplot_titles=("Bytes vs events (one dot = one day)", "D over time"))
    fig.add_trace(go.Scatter(x=dd["events"] / 1e6, y=gb(dd["bytes"]), mode="markers", name="days",
                             customdata=dd.index.strftime(HOVER_DAY),
                             hovertemplate="%{customdata}<br>%{x:,.1f} M events<br>%{y:,.1f} GiB<extra></extra>"),
                  row=1, col=1)
    xs = np.linspace(dd["events"].min(), dd["events"].max(), 50)
    fig.add_trace(go.Scatter(x=xs / 1e6, y=gb(icpt + slope * xs), mode="lines", line=dict(color="red"),
                             name=f"fit: {slope:.0f} B/event, R²={r2:.3f}", hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(x=xs / 1e6, y=gb(VENDOR_D * xs), mode="lines", line=dict(color="black", dash="dash"),
                             name=f"vendor: {VENDOR_D} B/event", hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(x=dd.index, y=dd["D"], mode="markers", name="daily D",
                             hovertemplate="%{x|" + HOVER_DAY + "}<br>%{y:,.0f} B/event<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Scatter(x=dd.index, y=dd["D"].rolling(14, min_periods=5).median(), mode="lines",
                             line=dict(color="red"), name="D, 14-day median",
                             hovertemplate="%{x|" + HOVER_DAY + "}<br>%{y:,.0f} B/event<extra>14-day median</extra>"),
                  row=1, col=2)
    fig.add_hline(y=VENDOR_D, line_dash="dash", line_color="black", row=1, col=2)
    weekends(fig, dd.index, row=1, col=2)
    fig.update_xaxes(title_text="events per day, millions", row=1, col=1)
    fig.update_yaxes(title_text="GiB per day", row=1, col=1)
    fig.update_yaxes(title_text="bytes per event", row=1, col=2)
    fig.update_xaxes(row=1, col=2, hoverformat=HOVER_DAY,
                     tickformatstops=[dict(dtickrange=[None, 40 * DAY_MS], value="%d.%m"),
                                      dict(dtickrange=[40 * DAY_MS, None], value="%m.%Y")])
    fig.update_layout(template="plotly_white", height=470, hovermode="closest",
                      legend=dict(orientation="h", yanchor="top", y=-0.2))
    out(fig, "cost_of_event")

    print(f"D (last 28 days median): {D_FACT:.1f} B/event; vendor formula overestimates by x{VENDOR_D / D_FACT:.2f}")
else:
    D_FACT = VENDOR_D
    print("Not enough days with events data; D falls back to the vendor value")

### How to read section 2

**Left — bytes vs events.** One dot = one day: X — events that day, Y — GiB on disk. Hover a dot to see its date.
- Dots lying close along the red line (R² close to 1) mean disk usage is driven by the number of events:
  forecasts through EPS are reliable, and the slope of the red line is the cost of one extra event.
- The gap between the red line and the black dashed one is how far the vendor's sizing formula is from reality.
- Dots far above the line are days with unusually heavy events; far below — incomplete days.

**Right — D over time** (bytes per event per day, red = 14-day median).
- Flat D with changing EPS — the flow changes, the event mix does not.
- A step in D with flat EPS — the event mix changed (heavier categories, longer raw messages), not the flow.
- Weekend dots may sit apart from weekdays: a different mix of sources at the weekend.

*Example (illustrative):* R² = 0.97, fit 95 B/event, vendor 200 → disk usage follows the event count and the vendor
formula overestimates disk about twice. D steps from ~95 to ~120 in one week while EPS stays flat → a heavy source was
added; check section 5 for the category whose cost grew.

## 3. Listpart bytes vs physical disk

`df used / sum(listpart bytes)` per snapshot. A stable ratio close to 1 means `listpart`
reflects real disk usage; the notebook uses this ratio to convert partition forecasts into physical free space.
A growing ratio means something other than partitions is eating the disk (staging, metadata, other files).

In [ ]:
if len(disk) and disk["partitions"].notna().any():
    disk["ratio"] = disk["used"] / disk["partitions"]
    RATIO = disk["ratio"].dropna().iloc[-1]

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                        subplot_titles=("Physical disk", "df used / sum of partitions"))
    hov = "%{x|%d.%m.%Y %H:%M}<br>%{y:,.2f} TiB<extra>%{fullData.name}</extra>"
    fig.add_trace(go.Scatter(x=disk.index, y=disk["used"] / TiB, mode="lines+markers", name="df used",
                             hovertemplate=hov), row=1, col=1)
    fig.add_trace(go.Scatter(x=disk.index, y=disk["partitions"] / TiB, mode="lines+markers",
                             name="sum of partitions", hovertemplate=hov), row=1, col=1)
    fig.add_hline(y=DISK_TOTAL / TiB, line_color="black", line_width=1, row=1, col=1,
                  annotation_text="disk total", annotation_position="top left")
    if LIMIT:
        fig.add_hline(y=LIMIT / TiB, line_color="red", line_dash="dash", row=1, col=1,
                      annotation_text=f"rotation limit {DBSPACE:.0%}", annotation_position="bottom left")
    fig.add_trace(go.Scatter(x=disk.index, y=disk["ratio"], mode="lines+markers", name="ratio",
                             hovertemplate="%{x|%d.%m.%Y %H:%M}<br>%{y:.3f}<extra>ratio</extra>"), row=1, col=2)
    fig.update_yaxes(title_text="TiB", row=1, col=1)
    fig.update_yaxes(range=[0.9, max(1.2, disk["ratio"].max() * 1.05)], row=1, col=2)
    fig.update_xaxes(tickformat="%d.%m", dtick=DAY_MS if len(disk) <= 60 else None)
    fig.update_layout(template="plotly_white", height=450, hovermode="closest",
                      legend=dict(orientation="h", yanchor="top", y=-0.15))
    out(fig, "disk")
    print(f"Latest: df used {disk['used'].iloc[-1]/TiB:.2f} TiB, partitions {disk['partitions'].iloc[-1]/TiB:.2f} TiB, ratio {RATIO:.3f}")
else:
    RATIO = 1.0
    print("No disk snapshots yet: ratio assumed 1.0")

### How to read section 3

One point = one collector run (snapshot), so the axis is snapshot time, not days of events.

**Left — physical disk.**
- `df used` (what the file system reports) and `sum of partitions` (what `listpart` reports) should run close to
  each other; both under the black *disk total* line.
- The red dashed line is the rotation limit from `DBSPACE`. **If the real partitions stay steadily above it, either
  rotation is not working or `DBSPACE` here differs from the tailcutter setting on the server** — check both.
- A flat `df used` while new days keep arriving means something deletes as much as is written every day: rotation works,
  just at the level where the line sits.

**Right — the ratio df used / partitions.**
- Stable and close to 1.00–1.05: `listpart` reflects real disk usage, and forecasts of partitions are forecasts of the disk.
- Growing: something other than partitions is filling the disk (staging, metadata, foreign files) — tailcutter will not
  remove it.

*Example (illustrative):* df used 9.4 TiB, partitions 9.3 TiB, ratio 1.01 and flat for a month → `listpart` can be
trusted. Ratio drifting 1.02 → 1.10 in two weeks → look for large files outside the partitions on the data volume.

## 4. Category dynamics

**Top:** share of each category in the daily flow. **Bottom:** linear trend per category
over the last `TREND_WINDOW` days, as % per month. Fast growers are the main drivers of future disk usage;
sudden drops usually mean a source stopped sending.

In [ ]:
ec = ev_cat.reindex(daily.index).loc[~daily["anomaly"]].dropna(how="all")
HAS_CAT = len(ec) >= 14

if HAS_CAT:
    share = ec.sum() / ec.sum().sum()
    main = share[share >= 0.01].sort_values(ascending=False).index
    grp = ec[main].copy()
    grp["other <1%"] = ec.drop(columns=main).sum(axis=1)

    fig = go.Figure()
    for col in grp.columns:
        fig.add_trace(go.Scatter(x=grp.index, y=grp[col], name=col, mode="lines", stackgroup="one",
                                 groupnorm="percent", line=dict(width=0.5),
                                 hovertemplate="%{y:.1f} %"))
    style(fig, "Category share of daily events", "% of daily events", height=520, slider=True)
    out(fig, "category_share")

    w = grp.tail(TREND_WINDOW)
    t = (w.index - w.index[0]).days.values
    trend = {}
    for c in tqdm(w.columns, desc="category trends"):
        k, _ = np.polyfit(t, w[c].values, 1)
        trend[c] = {"share_%": share.get(c, w[c].sum() / w.sum().sum()) * 100,
                    "growth_%/month": k * 30 / max(w[c].mean(), 1) * 100,
                    "Δevents/day per month": k * 30}
    trend = pd.DataFrame(trend).T.sort_values("growth_%/month")

    fig = go.Figure(go.Bar(x=trend["growth_%/month"], y=trend.index, orientation="h",
                           marker_color=np.where(trend["growth_%/month"] > 0, "crimson", "seagreen"),
                           customdata=np.c_[trend["share_%"], trend["Δevents/day per month"]],
                           hovertemplate="%{y}<br>growth %{x:+.1f} % per month<br>share %{customdata[0]:.1f} %"
                                         "<br>%{customdata[1]:+,.0f} events/day per month<extra></extra>"))
    fig.update_layout(template="plotly_white", height=max(380, 26 * len(trend)), xaxis_title="% per month",
                      title=f"Linear trend per category, last {len(w)} days ({w.index[0]:%d.%m.%Y} – {w.index[-1]:%d.%m.%Y})")
    out(fig, "category_growth")
    out_table(trend.round(1).sort_values("Δevents/day per month", ascending=False), "category_trends")
else:
    print("Not enough events data (need ≥14 days)")

### How to read section 4

**Category share** — each band is one category's share of the day's events, all bands add up to 100 %.
Hover a day to see every category's share on it; click a category in the legend to hide it, double-click to show it alone.
- A band that slowly widens — a category growing faster than the rest.
- A band that suddenly appears, jumps or disappears — a source was connected, started flooding or stopped sending.
  A disappearing band is the earliest sign of a collection problem.
- Categories under 1 % are summed into `other <1%`.

**Linear trend per category** — growth of each category over the last `TREND_WINDOW` days, % per month
(red = growing, green = shrinking). Hover a bar for its share and the absolute change in events per day.
Read it together with the share: +30 % per month of a 0.5 % category matters less than +5 % of a 40 % one.

**The table** `category_trends` — share, growth % per month and Δ events/day per month, sorted by the absolute change:
the top rows are the categories that drive the future flow.

*Example (illustrative):* Operating system keeps 40 % share but grows +6 % per month (+2 M events/day each month) — the
main driver. Proxy grows +25 % per month but has a 2 % share — watch, but it will not decide capacity.
A Firewall band that drops from 20 % to 3 % in one day → a firewall stopped sending; check its collection task.

## 5. Disk cost per category

NNLS regression `bytes = Σ bᵢ × eventsᵢ` (non-negative bytes per event per category).
`p5…p95` comes from bootstrap: a narrow interval means the coefficient is reliable.
With only a few weeks of data the intervals are wide — trust them after 3+ months of history.

In [ ]:
if HAS_CAT:
    yb = daily.loc[grp.index, "bytes"].values.astype(float)
    A = grp.values.astype(float)
    coef, _ = nnls(A, yb)
    mape_fit = np.mean(np.abs(yb - A @ coef) / yb)

    rng = np.random.default_rng(0)
    boot = np.array([nnls(A[i], yb[i])[0]
                     for i in tqdm([rng.integers(0, len(yb), len(yb)) for _ in range(500)], desc="bootstrap")])

    cost = pd.DataFrame({
        "share_%": (grp.sum() / grp.sum().sum() * 100).values,
        "B/event": coef,
        "p5": np.percentile(boot, 5, axis=0),
        "p95": np.percentile(boot, 95, axis=0),
        "GiB/day": gb(grp.tail(28).mean().values * coef),
    }, index=grp.columns).sort_values("GiB/day", ascending=False)

    base = grp.tail(28).mean().reindex(cost.index)
    fig = go.Figure(go.Bar(
        x=cost.index, y=cost["GiB/day"], marker_color="steelblue",
        error_y=dict(type="data", symmetric=False,
                     array=gb(base * (cost["p95"] - cost["B/event"])).values,
                     arrayminus=gb(base * (cost["B/event"] - cost["p5"])).values),
        customdata=np.c_[cost["B/event"], cost["p5"], cost["p95"], cost["share_%"]],
        hovertemplate="%{x}<br>%{y:,.1f} GiB/day<br>%{customdata[0]:,.0f} B/event "
                      "(p5 %{customdata[1]:,.0f} – p95 %{customdata[2]:,.0f})<br>share %{customdata[3]:.1f} %<extra></extra>"))
    fig.update_layout(template="plotly_white", height=450, yaxis_title="GiB/day",
                      title="Estimated disk usage per category (whiskers: p5…p95)")
    out(fig, "category_cost")
    print(f"Days in model: {len(yb)}, fit error (MAPE): {mape_fit:.2%}")
    out_table(cost.round(1), "category_cost")

### How to read section 5

**The chart** — how many GiB per day each category takes on disk, over the last 28 days. Whiskers are the p5…p95
range of the estimate; hover a bar for bytes per event and its range.
- A short whisker — the estimate is reliable and can be used to decide what to filter.
- A long whisker — too little history to separate this category from the others; trust it after 3+ months of data.
- `other <1%` merges small categories; its coefficient is usually unstable.

**The table** `category_cost`: share of events, bytes per event (`B/event`, `p5`, `p95`) and GiB per day.
A category with a small share but a high `B/event` is expensive per event — a candidate for filtering or tuning.
The fit error printed under the chart (MAPE) shows how well the categories explain the daily bytes: a few % is good.

*Example (illustrative):* DNS has 7 % of events but 180 B/event and 9 GiB/day with a short whisker — cutting its noise
pays off on disk. `Other` shows 900 B/event with p5 0 … p95 2 500 — not enough data, do not act on it yet.

## 6. Forecast models and backtest

Three models forecast daily bytes:

- **Linear** — linear trend + day-of-week effects over the last `TREND_WINDOW` days. Transparent, good for steady growth.
- **Holt-Winters** — exponential smoothing with damped trend and weekly seasonality. Adapts to level shifts faster.
- **SARIMAX** — seasonal ARIMA `SARIMAX_ORDER` × `SARIMAX_SEASONAL_ORDER` (weekly period) with drift,
  fitted on the last `SARIMAX_WINDOW` days (statsmodels, skipped if not installed).
  Models how day-to-day deviations carry over and the weekly cycle; the drift continues the average growth of the window.

Each model is trained without the last `BACKTEST_DAYS` days and scored on them.
The model with the lowest MAPE is used in sections 7–10.

In [ ]:
def model_linear(y, horizon, window=TREND_WINDOW):
    """Linear trend + day-of-week dummies. Returns yhat and residual sigma."""
    h = y.iloc[-window:]
    def X(idx, t0):
        t = (idx - t0).days.values.astype(float)
        dow = np.eye(7)[idx.dayofweek][:, 1:]
        return np.c_[np.ones(len(idx)), t, dow]
    t0 = h.index[0]
    beta, *_ = np.linalg.lstsq(X(h.index, t0), h.values, rcond=None)
    sigma = np.std(h.values - X(h.index, t0) @ beta)
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(X(fidx, t0) @ beta, index=fidx), sigma


def _hw_run(y, m, a, b, g, phi, horizon=0):
    """Additive Holt-Winters with damped trend; returns one-step fitted values and forecast."""
    n = len(y)
    l, tr = y[:m].mean(), (y[m:2 * m].mean() - y[:m].mean()) / m
    S = np.empty(n + m); S[:m] = y[:m] - l
    fit = np.empty(n)
    for i in range(n):
        fit[i] = l + phi * tr + S[i]
        l_new = a * (y[i] - S[i]) + (1 - a) * (l + phi * tr)
        tr = b * (l_new - l) + (1 - b) * phi * tr
        S[i + m] = g * (y[i] - l_new) + (1 - g) * S[i]
        l = l_new
    k = np.arange(1, horizon + 1)
    damp = np.cumsum(phi ** k)
    fc = l + damp * tr + S[n + (k - 1) % m]
    return fit, fc


def model_holt_winters(y, horizon, m=7):
    """Fit alpha/beta/gamma/phi by minimising one-step SSE."""
    v = y.values.astype(float)
    sc = v.mean()
    z = v / sc
    obj = lambda p: np.sum((z[m:] - _hw_run(z, m, *p)[0][m:]) ** 2)
    res = minimize(obj, x0=[0.3, 0.05, 0.1, 0.95], method="L-BFGS-B",
                   bounds=[(0.01, 1), (0.0, 0.5), (0.0, 1), (0.8, 0.999)])
    fit, fc = _hw_run(z, m, *res.x, horizon=horizon)
    sigma = np.std(z[m:] - fit[m:]) * sc
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(fc * sc, index=fidx), sigma


try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_SARIMAX = True
except ImportError:
    HAS_SARIMAX = False
    print("statsmodels not installed: SARIMAX is skipped (pip install statsmodels)")


def model_sarimax(y, horizon, window=SARIMAX_WINDOW,
                  order=SARIMAX_ORDER, seasonal_order=SARIMAX_SEASONAL_ORDER):
    """Seasonal ARIMA with drift on scaled values. Returns yhat and residual sigma.
    trend="c" in statsmodels SARIMAX acts on the differenced series, so with d=1 it is a drift (linear growth)."""
    h = y.iloc[-window:] if window else y
    sc = h.mean()
    z = h.values.astype(float) / sc
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # convergence warnings on short / flat series
        res = SARIMAX(z, order=order, seasonal_order=seasonal_order, trend="c").fit(disp=False)
    burn = order[1] + seasonal_order[1] * seasonal_order[3] + seasonal_order[3]   # start-up residuals are not errors
    sigma = np.std(res.resid[burn:]) * sc
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(res.forecast(horizon) * sc, index=fidx), sigma


MODELS = {"Linear": model_linear, "Holt-Winters": model_holt_winters}
if HAS_SARIMAX:
    MODELS["SARIMAX"] = model_sarimax

In [ ]:
# --- backtest on the last BACKTEST_DAYS days ---
train, test = y_bytes.iloc[:-BACKTEST_DAYS], y_bytes.iloc[-BACKTEST_DAYS:]
test_ok = ~daily["anomaly"].iloc[-BACKTEST_DAYS:].values   # score only real (non-outage) days

bt, bt_pred = {}, {}
for name, fn in tqdm(MODELS.items(), desc="backtest"):
    pred, _ = fn(train, BACKTEST_DAYS)
    err = (pred.values - test.values)[test_ok]
    bt_pred[name] = pred
    bt[name] = {"MAE, GiB/day": gb(np.mean(np.abs(err))),
                "MAPE, %": np.mean(np.abs(err) / test.values[test_ok]) * 100,
                "bias, GiB/day": gb(np.mean(err)),
                "sigma_bt": np.std(err)}
bt = pd.DataFrame(bt).T.sort_values("MAPE, %")
BEST = bt.index[0]

shown = y_bytes.iloc[-BACKTEST_DAYS * 3:]
outage = daily["anomaly"].reindex(shown.index).fillna(False).astype(bool).values   # interpolated, not real data
fig = go.Figure()
fig.add_trace(line(shown.index, gb(shown), "actual", "GiB", mode="lines+markers",
                   line=dict(color="black", width=1.5), marker=dict(size=5)))
fig.add_trace(line(shown.index[outage], gb(shown[outage]), "outage day, interpolated", "GiB", mode="markers",
                   marker=dict(symbol="x", size=10, color="red")))
for name, pr in bt_pred.items():
    fig.add_trace(line(pr.index, gb(pr), f"{name}: MAPE {bt.loc[name, 'MAPE, %']:.1f}%, "
                                         f"bias {bt.loc[name, 'bias, GiB/day']:+.1f} GiB/day", "GiB",
                       line=dict(width=3.5 if name == BEST else 1.5)))
weekends(fig, shown.index)
mark(fig, train.index[-1], "train end")
style(fig, f"Backtest on the last {BACKTEST_DAYS} days — best: {BEST}", "GiB/day", height=560, daily_ticks=True)
out(fig, "backtest")
out_table(bt.drop(columns="sigma_bt").round(2), "backtest")

### How to read the backtest chart

**What is on it**
- **Black line with dots** — actual GiB per day. Left of the dotted line *train end*: the end of the data the models
  were trained on (only the last weeks are drawn). Right of it: the last `BACKTEST_DAYS` days, hidden from the models.
- **Coloured lines** — what each model predicted for those hidden days, knowing only the data left of the dotted line.
  The thick line is the best model (lowest MAPE).
- **Red ×** — an outage day: its value is interpolated, not real data.
- **Grey bands** — weekends. Every day has a label; hover shows the date with the weekday and every line's value.

**How it is computed**
- The daily series is split in two: everything except the last `BACKTEST_DAYS` days is the training part,
  the last `BACKTEST_DAYS` days are the test part.
- Each model is fitted on the training part only and asked to forecast the test days; the forecast is compared with
  what actually happened.
- Outage days (section 1) are interpolated in the series, and in the test window they are **not scored**:
  an interpolated value is not a real observation, so it must not make a model look better or worse.
- The model with the lowest MAPE becomes the "best" one: its forecast drives sections 7–10,
  and its MAPE sets the width of the uncertainty band in section 7.

**The table under the chart** (one row per model, sorted by MAPE)
- `MAE, GiB/day` — mean absolute error: how many GiB per day the model misses on an average day, in either direction.
- `MAPE, %` — the same miss relative to the actual volume of that day. It compares models and periods of different
  scale; this is the ranking criterion.
- `bias, GiB/day` — mean of *forecast − actual*. Negative = the model under-predicts, positive = over-predicts.
  MAE says how big the misses are, bias says whether they lean one way. A large MAE with bias near 0 is random noise;
  bias close to MAE means the model is systematically off and its level is wrong.

**What to check, in this order**
1. **Level.** Does the best model run through the middle of the black line, or stay above / below it?
   Column `bias` in the table: negative = the model under-predicts. For capacity planning under-prediction is the
   dangerous direction — the disk fills and rotation starts earlier than forecast.
2. **Weekly rhythm.** The model's dips should line up with the actual weekend dips. If they do, the weekly cycle is
   captured. Misaligned dips mean the calendar pattern changed (holidays, shifts).
3. **Spikes.** Days far above every model. No model predicts one-off events: find what happened on those days
   (new source, incident, mass event) in sections 4–5. Spikes that keep repeating are a new normal, not noise.
4. **Size of the error.** MAPE × current daily volume = the typical miss per day in GiB.
   Under ~10 % — fit for planning; 10–20 % — plan with the band from section 7, not the line;
   above 20 % — the flow changed recently and the long-term forecast is unreliable.
5. **Agreement.** Models that are close in the test window can still diverge far apart in section 7.
   The backtest checks the next weeks only; it says nothing about a 180-day horizon.

**Example** *(illustrative numbers, not a real installation)*

> The training data shows a steady ~40 GiB/day that jumps to ~60 GiB/day three weeks before *train end*.
> In the test window:
> - **Linear** — MAPE 8 %, bias −2 GiB/day: follows the level and the weekend dips, slightly low. Best model.
> - **Holt-Winters** — MAPE 12 %, bias +3: the same shape, slightly high.
> - **SARIMAX** — MAPE 19 %, bias +9: above the actual on almost every day — it took the recent jump for a trend
>   and keeps climbing.
> - Two weekdays in the first test week reach ~85 GiB/day and no model saw them coming → look up in sections 4–5
>   which categories grew on those days.
>
> **Conclusion:** for the next weeks expect about 60 ± 5 GiB/day (8 % of 60), and keep a margin because the best
> model runs low. The jump in the training data is recent, so read the 180-day forecast in section 7 as a range,
> not a line, and re-check it in 2–3 weeks.

## 7. Daily volume forecast

Thick line = best model, dashed = other models (all as 7-day means). Shaded band = range between the models, widened by the best model's backtest error.
Read it as: *"if the flow keeps its current dynamics, a typical day will cost this much"*.
If the models diverge strongly, the history contains a recent regime change — trust the short-term part only
and re-check after 2–3 weeks.

In [ ]:
fc = {}
for name, fn in tqdm(MODELS.items(), desc="forecast"):
    fc[name] = fn(y_bytes, HORIZON)[0].clip(lower=0)

def band(fcs, mape):
    """Uncertainty band: range between models widened by the best model's backtest MAPE.
    Smoothed weekly, because capacity depends on sums of many days, not on single-day noise."""
    df_ = pd.DataFrame(fcs).rolling(7, min_periods=1).mean()
    return (df_.min(axis=1) * (1 - mape)).clip(lower=0), df_.max(axis=1) * (1 + mape)

FC = fc[BEST]
FC_LO, FC_HI = band(fc, bt.loc[BEST, "MAPE, %"] / 100)

fig = go.Figure()
fig.add_trace(line(daily.index, gb(daily["bytes"]), "actual", "GiB", line=dict(color="lightgrey", width=1)))
fig.add_trace(line(y_bytes.index, gb(y_bytes.rolling(7).mean()), "actual, 7-day mean", "GiB",
                   line=dict(color="black", width=1.5)))
fig.add_trace(go.Scatter(x=FC.index, y=gb(FC_HI), mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=FC.index, y=gb(FC_LO), mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor="rgba(70,130,180,0.18)", name="uncertainty band", hoverinfo="skip"))
for name, pr in fc.items():
    fig.add_trace(line(pr.index, gb(pr.rolling(7, min_periods=1).mean()), f"{name}, 7-day mean", "GiB",
                       line=dict(width=3 if name == BEST else 1.5, dash="solid" if name == BEST else "dash")))
mark(fig, last_day)
style(fig, f"Daily volume: history and {HORIZON}-day forecast (best: {BEST})", "GiB/day", height=520, slider=True)
out(fig, "volume_forecast")

now7 = y_bytes.tail(7).mean()
end30 = FC.tail(30).mean()
print(f"Now (7-day mean): {gb(now7):.0f} GiB/day ≈ {now7 / D_FACT / 86400:,.0f} EPS")
print(f"In {HORIZON} days ({BEST}): {gb(end30):.0f} GiB/day ≈ {end30 / D_FACT / 86400:,.0f} EPS "
      f"(band {gb(FC_LO.tail(30).mean()):.0f}…{gb(FC_HI.tail(30).mean()):.0f})")

In [ ]:
# --- EPS forecast (what the monitoring team sees in SIEM) ---
y_eps = daily["eps"].mask(daily["anomaly"])
if y_eps.notna().sum() >= 4 * 7:
    y_eps = y_eps.interpolate(limit_direction="both").loc[y_eps.first_valid_index():]
    p_eps = {k: f(y_eps, HORIZON)[0] for k, f in tqdm(MODELS.items(), desc="EPS forecast")}
    lo_e, hi_e = band(p_eps, bt.loc[BEST, "MAPE, %"] / 100)
    fig = go.Figure()
    fig.add_trace(line(y_eps.index, y_eps, "actual EPS", "EPS", line=dict(color="lightgrey", width=1)))
    fig.add_trace(line(y_eps.index, y_eps.rolling(7).mean(), "7-day mean", "EPS", line=dict(color="black")))
    fig.add_trace(go.Scatter(x=hi_e.index, y=hi_e, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=lo_e.index, y=lo_e, mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(70,130,180,0.18)", name="uncertainty band", hoverinfo="skip"))
    for k, pr in p_eps.items():
        fig.add_trace(line(pr.index, pr.rolling(7, min_periods=1).mean(), f"{k}, 7-day mean", "EPS",
                           line=dict(width=3 if k == BEST else 1.5, dash="solid" if k == BEST else "dash")))
    mark(fig, last_day)
    style(fig, "EPS: history and forecast", "events per second", height=520, slider=True)
    out(fig, "eps_forecast")
else:
    print("EPS forecast needs ≥4 weeks of events data")

### How to read section 7

**Daily volume** (and the same for **EPS**): light grey — actual days, black — 7-day mean of history; right of
*last complete day* — the forecast of every model as a 7-day mean (thick = best model from section 6) and the blue band.
- The band is the spread between the models widened by the best model's backtest error. **Plan with the band, not the
  line**: the upper edge is the pessimistic case for the disk.
- A narrow band — the models agree, the history is steady. A band that fans out — the models disagree about the future,
  usually because the history contains a recent step. Then trust the first 2–4 weeks and watch which line the actual
  follows.
- A straight steep line of `Linear` after a recent step in the history: it treats the step as a trend and may
  overestimate the long term; a flat `Holt-Winters` is the opposite, conservative case.
- Use *3 mo* to see the joint between history and forecast; hover shows each model's value on a given day.

The two printed lines give today's 7-day mean and the expected volume at the end of the horizon with the band.
EPS is derived as bytes / D — without events data D is the vendor value and EPS is only a rough estimate.

*Example (illustrative):* now 60 GiB/day; in 180 days Linear 95, Holt-Winters 62, SARIMAX 75, band 55…105 → the next
month is ~60–65 GiB/day for sure; the half-year can be anything from flat to +60 %. Size the disk for ~100 and re-check
monthly which line the actual follows.

## 8. Capacity: rotation start, retention depth, physical free space

Tailcutter keeps **sum of partitions ≤ DBSPACE × disk** by deleting the oldest days, and also drops days older than TTL.

- **Chart 1 — occupied space.** Grey dashed = what partitions would occupy without size-based rotation;
  blue = expected occupancy with rotation; red = limit. The point where grey crosses red is when automatic deletion starts.
- **Chart 2 — retention depth.** How many days of events will be kept on each future date.
  This is the main business metric: it falls when the flow grows.
- **Chart 3 — physical free space** (`df avail`), forecast = partitions forecast × `df/partitions` ratio,
  plus a plain linear trend of `df` snapshots for comparison.

In [ ]:
def project(stored_series, forecast, limit, ttl, other=0.0):
    """Simulate tailcutter on stored partitions + forecast. Returns frame per future day."""
    vals = np.r_[stored_series.values, forecast.values].astype(float)
    cs = np.r_[0.0, np.cumsum(vals)]
    n0 = len(stored_series)
    rows = []
    for j in range(n0, len(vals)):
        end = j + 1
        start_ttl = max(0, end - ttl)
        no_size_rot = cs[end] - cs[start_ttl] + other
        start = start_ttl
        if limit is not None:
            start = max(start, int(np.searchsorted(cs, cs[end] - (limit - other), side="left")))
        rows.append({"no_size_rotation": no_size_rot,
                     "occupied": cs[end] - cs[start] + other,
                     "depth_days": end - start})
    return pd.DataFrame(rows, index=forecast.index)

if LIMIT is None:
    print("No disk total (no df snapshot and DISK_TOTAL_OVERRIDE is None) — capacity section skipped")
else:
    proj = {k: project(stored_events, s, LIMIT, TTL_DAYS, stored_other)
            for k, s in tqdm({"mid": FC, "lo": FC_LO, "hi": FC_HI}.items(), desc="capacity")}
    P = proj["mid"]
    hit = P.index[P["no_size_rotation"] > LIMIT]
    HIT_DATE = hit[0] if len(hit) else None
    DAYS_TO_LIMIT = (HIT_DATE - last_day).days if HIT_DATE is not None else None
    hit_lo = proj["hi"].index[proj["hi"]["no_size_rotation"] > LIMIT]
    hit_hi = proj["lo"].index[proj["lo"]["no_size_rotation"] > LIMIT]

    # chart 1: occupied space
    fig = go.Figure()
    fig.add_trace(line(disk.index, disk["partitions"] / TiB, "partitions (actual)", "TiB", mode="lines+markers",
                       line=dict(color="black")))
    fig.add_trace(line(P.index, P["no_size_rotation"] / TiB, "without size rotation", "TiB",
                       line=dict(color="grey", dash="dash")))
    fig.add_trace(line(P.index, P["occupied"] / TiB, "with rotation (expected)", "TiB", line=dict(color="royalblue", width=3)))
    fig.add_hline(y=LIMIT / TiB, line_color="red", line_dash="dash",
                  annotation_text=f"limit {DBSPACE:.0%} = {LIMIT/TiB:.2f} TiB", annotation_position="bottom left")
    fig.add_hline(y=DISK_TOTAL / TiB, line_color="black", line_width=1,
                  annotation_text="disk total", annotation_position="top left")
    if HIT_DATE is not None:
        fig.add_shape(type="line", x0=HIT_DATE, x1=HIT_DATE, y0=0, y1=1, yref="paper", line=dict(color="red", width=1))
        fig.add_annotation(x=HIT_DATE, y=LIMIT / TiB, text=f"rotation starts ~{HIT_DATE:%d.%m.%Y}", showarrow=True,
                           arrowhead=2, font=dict(color="red"))
    mark(fig, last_day)
    style(fig, "Occupied by partitions", "TiB", slider=True)
    out(fig, "capacity_occupied")

    # chart 2: retention depth
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=P.index, y=proj["lo"]["depth_days"], mode="lines", line=dict(width=0),
                             showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=P.index, y=proj["hi"]["depth_days"], mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(65,105,225,0.18)", name="uncertainty band", hoverinfo="skip"))
    fig.add_trace(line(P.index, P["depth_days"], f"expected ({BEST})", "days", line=dict(color="royalblue", width=3)))
    fig.add_hline(y=TTL_DAYS, line_color="grey", line_dash="dot", annotation_text=f"TTL {TTL_DAYS}",
                  annotation_position="top left")
    mark(fig, last_day, f"now: {len(stored_events)} days stored")
    style(fig, "Retention depth forecast: days of events kept", "days kept", slider=True)
    out(fig, "capacity_retention")

    # chart 3: physical free space
    phys_avail = DISK_TOTAL - P["occupied"] * RATIO
    fig = go.Figure()
    fig.add_trace(line(disk.index, disk["avail"] / TiB, "df avail (actual)", "TiB", mode="lines+markers",
                       line=dict(color="black")))
    fig.add_trace(line(P.index, phys_avail / TiB, "forecast from partitions", "TiB", line=dict(color="royalblue", width=3)))
    if len(disk) >= 3:
        td = (disk.index - disk.index[0]).total_seconds().values / 86400
        k, c0 = np.polyfit(td, disk["avail"].values, 1)
        tf = (P.index - disk.index[0]).total_seconds().values / 86400
        fig.add_trace(line(P.index, (c0 + k * tf) / TiB, f"linear trend of df = if rotation does not work "
                                                         f"({gb(k):+.0f} GiB/day)", "TiB",
                           line=dict(color="orange", dash="dash")))
    fig.add_hline(y=DISK_TOTAL * (1 - DBSPACE) / TiB, line_color="red", line_dash="dash",
                  annotation_text="free space at limit", annotation_position="top left")
    fig.add_hline(y=0, line_color="black", line_width=1)
    mark(fig, last_day)
    style(fig, "Physical free space", "TiB", slider=True)
    out(fig, "capacity_free_space")

    out_table(pd.DataFrame({
        "expected, TiB": (P["occupied"] / TiB).round(2),
        "without size rotation, TiB": (P["no_size_rotation"] / TiB).round(2),
        "retention, days": P["depth_days"],
        "retention low, days": proj["hi"]["depth_days"],
        "retention high, days": proj["lo"]["depth_days"],
        "free space forecast, TiB": (phys_avail / TiB).round(2),
    }), "capacity_projection", show=False)

    print(f"Disk {DISK_TOTAL/TiB:.2f} TiB, limit {LIMIT/TiB:.2f} TiB, partitions now {stored_total/TiB:.2f} TiB "
          f"({stored_total/LIMIT:.0%} of limit)")
    if HIT_DATE is not None:
        rng_txt = (f"{hit_lo[0]:%d.%m.%Y}" if len(hit_lo) else "?") + "…" + \
                  (f"{hit_hi[0]:%d.%m.%Y}" if len(hit_hi) else f"after {HORIZON} days")
        print(f"Size rotation starts ~{HIT_DATE:%d.%m.%Y} (in {DAYS_TO_LIMIT} days; band {rng_txt})")
    else:
        print(f"Size limit is not reached within {HORIZON} days")
    print(f"Retention depth: now {len(stored_events)} days → in {HORIZON} days ~{P['depth_days'].iloc[-1]} days "
          f"(band {proj['hi']['depth_days'].iloc[-1]}…{proj['lo']['depth_days'].iloc[-1]})")

### How to read section 8

**Occupied by partitions.** Black — actual sum of partitions at each snapshot; grey dashed — what partitions would
occupy if tailcutter deleted only by TTL; blue — expected occupancy with size rotation; red — the limit
`DBSPACE × disk`; black solid — the whole disk.
- Where the grey line crosses red, size rotation starts deleting the oldest days (red vertical line and date).
- **Black already above red** — the limit is exceeded now. Either rotation does not work, or `DBSPACE` in the notebook
  differs from the real tailcutter setting; the "starts in N days" message is meaningless in that case.

**Retention depth** — how many days of events will be kept on each future date (blue), with the band from the
forecast. This is the business number: compare it with the required retention. It falls as the flow grows.
The vertical line shows how many days are stored now.

**Physical free space** — blue: expected `df avail` with rotation working; orange dashed: the plain trend of `df`
snapshots, i.e. the scenario "rotation does not work" — where it crosses zero the disk is full.
The orange line is only meaningful after a few weeks of snapshots.

**The table** `capacity_projection` (saved, not shown): per future day — occupancy with and without size rotation,
retention depth with its band and the free-space forecast.

*Example (illustrative):* limit 8.5 TiB, partitions 7.9 TiB, grey crosses red on 14.03 → in ~5 months rotation starts;
retention falls from 240 to ~150 days by the end of the horizon, band 120…190. If the required retention is 180 days,
the decision (more disk, filtering, or accepting less) is due before the band's lower edge crosses 180.

## 9. What-if: flow changes

**Table:** a uniform change of the whole flow → daily volume and steady-state retention depth.
**Function `what_if_category`:** change of one category, priced with the coefficients from section 5.

In [ ]:
eps_now = now7 / D_FACT / 86400
rows = []
for pct in tqdm([-50, -30, -20, -10, 0, 10, 20, 30, 50, 100], desc="scenarios"):
    day_b = now7 * (1 + pct / 100)
    r = {"flow change, %": pct,
         "ΔEPS": eps_now * pct / 100,
         "GiB/day": gb(day_b),
         "ΔGiB/day": gb(day_b - now7)}
    if LIMIT:
        r["retention, days"] = min(TTL_DAYS, int((LIMIT - stored_other) // day_b))
    rows.append(r)
out_table(pd.DataFrame(rows).set_index("flow change, %").round(1), "what_if")


def what_if_category(category, pct):
    """Daily volume change if one category changes by pct %."""
    if not HAS_CAT or category not in cost.index:
        print(f"Not in model: {category}. Available: {list(cost.index) if HAS_CAT else []}")
        return
    ev = grp[category].tail(28).mean() * pct / 100
    r = cost.loc[category]
    msg = (f"{category} {pct:+}% → {ev:+,.0f} events/day → {gb(ev * r['B/event']):+.1f} GiB/day "
           f"(band {gb(ev * r['p5']):+.1f}…{gb(ev * r['p95']):+.1f})")
    if LIMIT:
        new = now7 + ev * r["B/event"]
        msg += f"; retention ~{min(TTL_DAYS, int((LIMIT - stored_other) // new))} days"
    print(msg)


if HAS_CAT:
    what_if_category(cost.drop(index="other <1%", errors="ignore").index[0], -30)

### How to read section 9

**The table** `what_if`: one row per uniform change of the whole flow. `ΔEPS` and `ΔGiB/day` — the change against today's
7-day mean; `retention, days` — the steady-state depth once the disk is at the limit with that flow
(capped at `TTL_DAYS`).
- Read it as "what does +X % of flow cost in days of retention" — the trade-off between connecting new sources and
  keeping history. The negative rows show what filtering buys.

**`what_if_category("<category>", pct)`** prints the same for one category, priced with its own bytes per event from
section 5 and its p5…p95 range. Call it in a new cell for any category from the section 5 table.

*Example (illustrative):* row +20 % → 72 GiB/day, retention 150 days instead of 180: connecting the planned sources
costs a month of history. `what_if_category("DNS server", -50)` → −4.5 GiB/day, retention ~+12 days: filtering half
of DNS buys back almost two weeks.

## 10. Daily summary

`OK` — nothing to do. `WARN` — look at the referenced section today. `CRIT` — act now (see METHODOLOGY.md).

In [ ]:
status = []
def add(level, check, value, section):
    status.append({"level": level, "check": check, "value": value, "see": section})

# data freshness
age_h = (pd.Timestamp.now() - last_snap).total_seconds() / 3600
add("OK" if age_h < 26 else "WARN", "latest snapshot age", f"{age_h:.0f} h", "1")
ev_last = daily["events"].last_valid_index()
add("OK" if ev_last == last_day else "WARN", "events collected up to",
    f"{ev_last:%d.%m.%Y}" if ev_last is not None else "none", "1")

# yesterday volume vs same weekday baseline (last 4 same weekdays)
same = daily["bytes"].loc[daily.index.dayofweek == last_day.dayofweek].iloc[-5:-1]
base = same.median()
yb_ = daily["bytes"].iloc[-1]
dev = (yb_ - base) / base if base else np.nan
add("OK" if abs(dev) < WARN_VOLUME_DEV else "WARN", f"volume {last_day:%d.%m} vs same weekday",
    f"{gb(yb_):.0f} GiB ({dev:+.0%})", "7")

# per-category deviation (source dropout or flood)
if HAS_CAT and last_day in ev_cat.index:
    hist_c = ev_cat.loc[ev_cat.index.dayofweek == last_day.dayofweek].iloc[-5:-1].median()
    cur_c = ev_cat.loc[last_day]
    rel = ((cur_c - hist_c) / hist_c.replace(0, np.nan)).dropna()
    rel = rel[hist_c.reindex(rel.index) > ev_cat.loc[last_day].sum() * 0.001]   # ignore tiny categories
    bad = rel[rel.abs() >= WARN_CATEGORY_DEV]
    add("OK" if bad.empty else "WARN", "categories off baseline",
        ", ".join(f"{k} {v:+.0%}" for k, v in bad.sort_values().items()) or "none", "4")

# D drift
if HAS_D and pd.notna(daily["D"].iloc[-1]):
    d_dev = daily["D"].iloc[-1] / D_FACT - 1
    add("OK" if abs(d_dev) < WARN_D_DRIFT else "WARN", "D yesterday vs 28-day median",
        f"{daily['D'].iloc[-1]:.0f} B/event ({d_dev:+.0%})", "2")

# capacity
if LIMIT:
    occ = stored_total / LIMIT
    add("CRIT" if occ > 1.02 else ("WARN" if occ > 0.95 else "OK"), "partitions / limit", f"{occ:.0%}", "8")
    if DAYS_TO_LIMIT is not None:
        add("WARN" if DAYS_TO_LIMIT < WARN_DAYS_TO_LIMIT else "OK", "size rotation starts in",
            f"{DAYS_TO_LIMIT} days (~{HIT_DATE:%d.%m.%Y})", "8")
    add("OK", "retention now → horizon", f"{len(stored_events)} → {P['depth_days'].iloc[-1]} days", "8")
    free = disk["avail"].iloc[-1] / DISK_TOTAL
    add("CRIT" if free < 0.05 else ("WARN" if free < 1 - DBSPACE else "OK"), "physical free space",
        f"{free:.1%} ({disk['avail'].iloc[-1]/TiB:.2f} TiB)", "8")

# rotation health
if "tailcutter_cron" in health:
    add("OK" if health["tailcutter_cron"] == 1 else "CRIT", "tailcutter in cron",
        "yes" if health["tailcutter_cron"] == 1 else "NO", "METHODOLOGY")
if "tailcutter_log_age_min" in health:
    a = health["tailcutter_log_age_min"]
    add("OK" if 0 <= a < WARN_TC_LOG_AGE else "WARN", "tailcutter log age",
        "no log" if a < 0 else f"{a:.0f} min", "METHODOLOGY")

add("OK", "best forecast model", f"{BEST} (backtest MAPE {bt.loc[BEST, 'MAPE, %']:.1f}%)", "6")

summary = pd.DataFrame(status)
out_table(summary.set_index("level"), "summary", show=False)
colors = {"OK": "background-color:#d9f2d9", "WARN": "background-color:#fff2b3", "CRIT": "background-color:#f8c6c6"}
sty = summary.style
styler_map = getattr(sty, "map", None) or sty.applymap     # pandas < 2.1 has only applymap
display(styler_map(lambda v: colors.get(v, ""), subset=["level"]).hide(axis="index"))

### How to read section 10

One row per check: `level` (OK / WARN / CRIT), what was checked, its value and the section to open (`see`).
Go top-down through every row that is not OK; the meaning of each check and what to do is in `METHODOLOGY.md`, section 4.
A CRIT that contradicts another row (for example *partitions / limit* CRIT while the disk is stable and tailcutter
runs) usually means a setting in section 0 does not match the server — check it before acting.
The table is also saved as `summary_<day>.xlsx`, so the status of different days can be compared.